# 直接偏好优化：数据准备

上一章，我们用 SFT 让模型学会了按照指令对话。但是 SFT 有一个局限：它只能告诉模型怎么回答，但是不能告诉模型什么是更好的答案。

比如两个答案都正确，但是一个简洁、一个啰嗦；一个礼貌、一个生硬。SFT 无法区分。

实践中，研究者们发现，可以通过在微调中告诉模型那个答案更合理，让模型学会更以更准确、更诚实、更安全的方式回答问题。这种微调技术，称为**偏好优化**（Preference Optimization, PFT）。通常叠加在 SFT 的基础上，进一步优化模型输出。

当前**偏好优化**主要包括以下方式：

* **基于人类反馈的强化学习**（Reinforcement Learning from Human Feedback, RLHF）：训练一个奖励模型，使用强化学习算法来优化模型；
* **直接偏好优化**（Direct Preference Optimization, DPO）：直接使用带偏好的数据对训练模型；
* **推理时偏好优化**（Test-Time Perference Optimization, TPO）：在推理阶段通过特定策略和算法（比如引导解码、拒绝采样）来调整模型输出。

## 直接偏好采样

**直接偏好优化**（Direct Preference Optimization, DPO）是让模型学会**比较**的技术：给模型看两个不同的正确答案，并告诉它哪个更好。

### 数据格式：

DPO 的数据格式是**三元组**：

* **提示**（Prompt）：用户的提问或者命令；
* **接受**（Chosen）：更合理的答案；
* **拒绝**（Rejected）：较差的答案。

例如：

```
提示：ROMEO:
     Hello, Juliet!
接受：JULIET:
     Hello, Romeo! Nice to meet you!
拒绝：JULIET:
     Hi!
```

### 数据来源

和 SFT 数据一样，我们依然从**微型莎士比亚剧本**（tinyshakespeare）数据集的**测试集**中提取对话样本作为**提示**和**接受答案**。而**拒绝答案**则采用相同的提示，由模型续写的结果。


In [1]:
import pickle
import re
from itertools import pairwise

import numpy as np

from src.core import Tensor
from src.gpt import *

In [2]:
np.random.seed(42)

## 数据提取

### 提取接受答案

相同的**提取函数** extract_turns 从莎士比亚剧本中提取测试集（最后 10% 的数据）的所有台词。

In [3]:
def extract_turns(filename, split=0.9, max_turn_chars=100):
    with open(filename, encoding="utf-8") as f:
        text = f.read()

    blocks = re.split(r"\n\s*\n", text.strip())
    turns = []

    split = int(len(blocks) * split)
    for block in blocks[split:]:
        lines = block.strip("\n").split("\n")
        if not lines:
            continue

        m = re.compile(r"^([A-Z][A-Za-z' ]{0,30}):\s*$").match(lines[0].strip())
        if not m:
            continue

        speaker = m.group(1).strip()
        content = " ".join(line.strip() for line in lines[1:] if line.strip())
        if not content:
            continue

        turns.append((speaker, content[:max_turn_chars]))

    return turns

### 生成拒绝答案

函数 generate_rejected_batch 用预训练模型批量生成**拒绝答案**：

* **批量生成**：同时处理多个提示，提高效率；
* **温度采样**：设置采用温度为 0.8，控制随机性，值越大越随机；
* **top-k采样**：设置 top-k 只从概率最高的 20 个词元中采样，避免生成乱码；
* **填充**：用 pad_token 把短序列补齐到上下文尺寸。

In [4]:
def generate_rejected_batch(model, prompts, lens, pad_token=0, temperature=0.8, top_k=20):
    sequences = [list(p) for p in prompts]

    with Tensor.no_grad():
        for _ in range(max(lens)):
            windows = [(([pad_token] * CONTEXT_SIZE) + seq)[-CONTEXT_SIZE:] for seq in sequences]
            batch = np.array(windows, dtype=np.int64)
            logits = model(Tensor(batch))
            next_logits = logits.data[:, -1, :] / temperature

            k = min(top_k, next_logits.shape[-1])
            for row in range(len(prompts)):
                row_logits = next_logits[row]
                threshold = np.partition(row_logits, -k)[-k]
                row_logits = np.where(row_logits < threshold, -np.inf, row_logits)
                probs = np.exp(row_logits - np.max(row_logits))
                probs /= np.sum(probs)
                sequences[row].append(int(np.random.choice(len(probs), p=probs)))

    return [seq[len(prompts[i]): len(prompts[i]) + lens[i]] for i, seq in enumerate(sequences)]

## 超参数

### 模型参数

所有用到的超参数保持和预训练一致。

In [5]:
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

## 构建样本

### 加载模型

In [6]:
DATA_FILE = "../../../../data/tinyshakespeare.txt"
MODEL_FILE = "../../../../data/tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
trainer = GPTTrainer(model, None, None)
trainer.load(MODEL_FILE)
model.eval()

### 提取样本

从数据集的测试集构建（提示，接受，拒绝）数据组：

* **配对**：和 SFT 一样，提取相邻的台词对，作为提示和接受答案；
* **生成拒绝答案**：用预训练模型为每个提示生成一个输出作为拒绝答案；
* **过滤**：去掉拒绝答案为空，或者和接受答案一致的数据组。

In [7]:
pairs = list(pairwise(extract_turns(DATA_FILE)))
np.random.shuffle(pairs)
pairs = pairs[:512]

samples = []
for start in range(0, len(pairs), 64):
    chunk = pairs[start: start + 64]

    prompt_tokens = [dataset.encode(f"{a}:\n{t}\n") for (a, t), (_, _) in chunk]
    chosen_tokens = [dataset.encode(f"{b}:\n{t}\n") for (_, _), (b, t) in chunk]

    lens = [len(c) for c in chosen_tokens]
    rejected_tokens = generate_rejected_batch(model, prompt_tokens, lens)

    for p, c, r in zip(prompt_tokens, chosen_tokens, rejected_tokens):
        if len(r) == 0 or r == c:
            continue
        samples.append((p, c, r))

    print(f"generated {len(samples)}/{len(pairs)} DPO samples")

generated 64/512 DPO samples
generated 128/512 DPO samples
generated 192/512 DPO samples
generated 256/512 DPO samples
generated 320/512 DPO samples
generated 384/512 DPO samples
generated 448/512 DPO samples
generated 512/512 DPO samples


### 保存样本

同样用 `pickle.dump()` 保存到文件。

In [8]:
DPO_SAMPLES = "../../../../data/dpo-samples.pkl"

with open(DPO_SAMPLES, "wb") as f:
    pickle.dump(samples, f)
print(f"saved {len(samples)} DPO samples to {DPO_SAMPLES}")

saved 512 DPO samples to ../../../../data/dpo-samples.pkl


### 验证样本

查看第一个样本，确认格式正确。

In [9]:
sample = samples[0]
print(f"{dataset.decode(sample[0])}\n{dataset.decode(sample[1])}\n{dataset.decode(sample[2])}")

SEBASTIAN:
Foul weather?

ANTONIO:
Very foul.


AUFIDIUS:
Dewome, a


本章，我们任然依靠微型莎士比亚剧本数据集的测试集，构建了 512 个 DPO 训练数据：每个样本包括一个**提示**、一个**接受答案**（真实台词）和一个**拒绝答案**（模型输出）。

下一章，我们将用这些样本对 SFT 模型进行**直接偏好优化**训练，让模型学会从多个答案中选出更好的一个。

## 课后练习

